<center> <img src = https://raw.githubusercontent.com/AndreyRysistov/DatasetsForPandas/main/hh%20label.jpg alt="drawing" style="width:400px;">

# <center> Проект: Анализ резюме из HeadHunter
   

исходный датабэйс : https://drive.google.com/file/d/1F-5PEOsRqxQCYm1C9bHgsfbDAyaub6Ko/view?usp=sharing

графики сохраняются в *.png

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import re

# Исследование структуры данных

1. Прочитайте данные с помощью библиотеки Pandas. Совет: перед чтением обратите внимание на разделитель внутри файла. 

In [ ]:
#ваш код здесь

# Замените путь к файлу на актуальный путь к .csv файлу
file_path = '/home/sexxlexx/Desktop/SKILL/dst-3.0_16_1_hh_database.csv'

# Чтение данных с указанием разделителя ";"
df = pd.read_csv(file_path, sep=";")

# Вывод размерности таблицы
print(df.shape)  # Вернет кортеж (число строк, число столбцов)

2. Выведите несколько первых (последних) строк таблицы, чтобы убедиться в том, что ваши данные не повреждены. Ознакомьтесь с признаками и их структурой.

In [ ]:
#ваш код здесь

# Вывод первых 5 строк
print("Первые 5 строк:")
print(df.head())

# Вывод последних 5 строк
print("\nПоследние 5 строк:")
print(df.tail())

# Вывод признаков (столбцов)
print("\nПризнаки (столбцы):")
print(df.columns)

3. Выведите основную информацию о числе непустых значений в столбцах и их типах в таблице.

4. Обратите внимание на информацию о числе непустых значений.

In [ ]:
#ваш код здесь
print("Информация о данных:")
df.info()

5. Выведите основную статистическую информацию о столбцах.


In [ ]:
#ваш код здесь
# Вывод основной статистической информации для всех типов данных
print("\nСтатистическая информация для всех столбцов:")
print(df.describe(include='all'))

# Преобразование данных

1. Начнем с простого - с признака **"Образование и ВУЗ"**. Его текущий формат это: **<Уровень образования год выпуска ВУЗ специальность...>**. Например:
* Высшее образование 2016 Московский авиационный институт (национальный исследовательский университет)...
* Неоконченное высшее образование 2000  Балтийская государственная академия рыбопромыслового флота…
Нас будет интересовать только уровень образования.

Создайте с помощью функции-преобразования новый признак **"Образование"**, который должен иметь 4 категории: "высшее", "неоконченное высшее", "среднее специальное" и "среднее".

Выполните преобразование, ответьте на контрольные вопросы и удалите признак "Образование и ВУЗ".

Совет: обратите внимание на структуру текста в столбце **"Образование и ВУЗ"**. Гарантируется, что текущий уровень образования соискателя всегда находится в первых 2ух слов и начинается с заглавной буквы. Воспользуйтесь этим.

*Совет: проверяйте полученные категории, например, с помощью метода unique()*


In [ ]:
#ваш код здесь
# Функция для классификации уровня образования
def classify_education(education_str):
    # Проверяем, что значение не NaN и является строкой
    if pd.isna(education_str) or not isinstance(education_str, str):
        return 'неизвестно'
    
    # Берем первые три слова
    first_words = ' '.join(education_str.split()[:3])
    
    # Логика определения уровня образования
    if 'Высшее образование' in first_words:
        return 'высшее'
    elif 'Неоконченное высшее образование' in first_words:
        return 'неоконченное высшее'
    elif 'Среднее специальное образование' in first_words:
        return 'среднее специальное'
    elif 'Среднее образование' in first_words:
        return 'среднее'
    
    # Если ничего не подошло
    return 'неизвестно'

# Проверяем наличие столбца "Образование и ВУЗ"
if 'Образование и ВУЗ' not in df.columns:
    raise KeyError("Столбец 'Образование и ВУЗ' отсутствует в данных!")

# Применяем функцию к столбцу «Образование и ВУЗ»
df['Образование'] = df['Образование и ВУЗ'].apply(classify_education)

# Проверяем уникальные значения в новом столбце
print("Уникальные значения в столбце 'Образование':", df['Образование'].unique())

# Удаляем старый столбец «Образование и ВУЗ»
df.drop(columns=['Образование и ВУЗ'], inplace=True)

# Проверяем размерность таблицы после удаления столбца
print(f"Размерность таблицы после обработки: {df.shape}")

# Выводим первые несколько строк результата
print(df.head())

2. Теперь нас интересует столбец **"Пол, возраст"**. Сейчас он представлен в формате **<Пол , возраст , дата рождения >**. Например:
* Мужчина , 39 лет , родился 27 ноября 1979 
* Женщина , 21 год , родилась 13 января 2000
Как вы понимаете, нам необходимо выделить каждый параметр в отдельный столбец.

Создайте два новых признака **"Пол"** и **"Возраст"**. При этом важно учесть:
* Признак пола должен иметь 2 уникальных строковых значения: 'М' - мужчина, 'Ж' - женщина. 
* Признак возраста должен быть представлен целыми числами.

Выполните преобразование, ответьте на контрольные вопросы и удалите признак **"Пол, возраст"** из таблицы.

*Совет: обратите внимание на структуру текста в столбце, в части на то, как разделены параметры пола, возраста и даты рождения между собой - символом ' , '. 
Гарантируется, что структура одинакова для всех строк в таблице. Вы можете воспользоваться этим.*


In [ ]:
#ваш код здесь
# Функция для извлечения пола
def extract_gender(value):
    if pd.isna(value):
        return None
    if 'Мужчина' in value:
        return 'М'
    elif 'Женщина' in value:
        return 'Ж'
    return None

# Функция для извлечения возраста
def extract_age(value):
    if pd.isna(value):
        return None
    try:
        age = int(value.split(',')[1].strip().split(' ')[0])  # Извлекаем возраст
        return age
    except:
        return None

# Создаем новые столбцы "Пол" и "Возраст"
df['Пол'] = df['Пол, возраст'].apply(extract_gender)
df['Возраст'] = df['Пол, возраст'].apply(extract_age)

# Удаляем старый столбец "Пол, возраст"
df.drop(columns=['Пол, возраст'], inplace=True)

3. Следующим этапом преобразуем признак **"Опыт работы"**. Его текущий формат - это: **<Опыт работы: n лет m месяцев, периоды работы в различных компаниях…>**. 

Из столбца нам необходимо выделить общий опыт работы соискателя в месяцах, новый признак назовем "Опыт работы (месяц)"

Для начала обсудим условия решения задачи:
* Во-первых, в данном признаке есть пропуски. Условимся, что если мы встречаем пропуск, оставляем его как есть (функция-преобразование возвращает NaN)
* Во-вторых, в данном признаке есть скрытые пропуски. Для некоторых соискателей в столбце стоит значения "Не указано". Их тоже обозначим как NaN (функция-преобразование возвращает NaN)
* В-третьих, нас не интересует информация, которая описывается после указания опыта работы (периоды работы в различных компаниях)
* В-четвертых, у нас есть проблема: опыт работы может быть представлен только в годах или только месяцах. Например, можно встретить следующие варианты:
    * Опыт работы 3 года 2 месяца…
    * Опыт работы 4 года…
    * Опыт работы 11 месяцев…
    * Учитывайте эту особенность в вашем коде

Учитывайте эту особенность в вашем коде

В результате преобразования у вас должен получиться столбец, содержащий информацию о том, сколько месяцев проработал соискатель.
Выполните преобразование, ответьте на контрольные вопросы и удалите столбец **"Опыт работы"** из таблицы.


In [ ]:
#ваш код здесь
# Функция для преобразования опыта работы в месяцы
def get_experience(arg):
    # Обработка пропусков и скрытых пропусков
    if pd.isna(arg) or arg == 'Не указано':
        return None
    
    # Словари для определения годов и месяцев
    year_words = ['год', 'года', 'лет']
    month_words = ['месяц', 'месяца', 'месяцев']
    
    # Разделяем строку на слова, учитываем только первые 7 слов
    arg_splitted = arg.split(' ')[:7]
    years = 0
    months = 0
    
    # Перебираем слова и находим значения лет и месяцев
    for index, item in enumerate(arg_splitted):
        if item in year_words:
            years = int(arg_splitted[index - 1])  # Значение перед словом "год", "года", "лет"
        if item in month_words:
            months = int(arg_splitted[index - 1])  # Значение перед словом "месяц", "месяца", "месяцев"
    
    return int(years * 12 + months)

# Применяем функцию к столбцу "Опыт работы"
df['Опыт работы (месяц)'] = df['Опыт работы'].apply(get_experience)

# Удаляем исходный столбец "Опыт работы"
df.drop(columns=['Опыт работы'], inplace=True)

4. Хорошо идем! Следующий на очереди признак "Город, переезд, командировки". Информация в нем представлена в следующем виде: **<Город , (метро) , готовность к переезду (города для переезда) , готовность к командировкам>**. В скобках указаны необязательные параметры строки. Например, можно встретить следующие варианты:

* Москва , не готов к переезду , готов к командировкам
* Москва , м. Беломорская , не готов к переезду, не готов к командировкам
* Воронеж , готов к переезду (Сочи, Москва, Санкт-Петербург) , готов к командировкам

Создадим отдельные признаки **"Город"**, **"Готовность к переезду"**, **"Готовность к командировкам"**. При этом важно учесть:

* Признак **"Город"** должен содержать только 4 категории: "Москва", "Санкт-Петербург" и "город-миллионник" (их список ниже), остальные обозначьте как "другие".

    Список городов-миллионников:
    
   <code>million_cities = ['Новосибирск', 'Екатеринбург','Нижний Новгород','Казань', 'Челябинск','Омск', 'Самара', 'Ростов-на-Дону', 'Уфа', 'Красноярск', 'Пермь', 'Воронеж','Волгоград']
    </code>
    Инфорация о метро, рядом с которым проживает соискатель нас не интересует.
* Признак **"Готовность к переезду"** должен иметь два возможных варианта: True или False. Обратите внимание, что возможны несколько вариантов описания готовности к переезду в признаке "Город, переезд, командировки". Например:
    * … , готов к переезду , …
    * … , не готова к переезду , …
    * … , готова к переезду (Москва, Санкт-Петербург, Ростов-на-Дону)
    * … , хочу переехать (США) , …
    
    Нас интересует только сам факт возможности или желания переезда.
* Признак **"Готовность к командировкам"** должен иметь два возможных варианта: True или False. Обратите внимание, что возможны несколько вариантов описания готовности к командировкам в признаке "Город, переезд, командировки". Например:
    * … , готов к командировкам , … 
    * … , готова к редким командировкам , …
    * … , не готов к командировкам , …
    
    Нас интересует только сам факт готовности к командировке.
    
    Еще один важный факт: при выгрузки данных у некоторых соискателей "потерялась" информация о готовности к командировкам. Давайте по умолчанию будем считать, что такие соискатели не готовы к командировкам.
    
Выполните преобразования и удалите столбец **"Город, переезд, командировки"** из таблицы.

*Совет: обратите внимание на то, что структура текста может меняться в зависимости от указания ближайшего метро. Учите это, если будете использовать порядок слов в своей программе.*


In [ ]:
#ваш код здесь
# Список городов-миллионников
million_cities = ['Новосибирск', 'Екатеринбург', 'Нижний Новгород', 'Казань',
                  'Челябинск', 'Омск', 'Самара', 'Ростов-на-Дону', 'Уфа',
                  'Красноярск', 'Пермь', 'Воронеж', 'Волгоград']

# Функция для извлечения города
def get_city(arg):
    city = arg.split(' , ')[0]  # Извлекаем город
    if city in ['Москва', 'Санкт-Петербург']:
        return city
    elif city in million_cities:
        return 'город миллионник'
    else:
        return 'другие'

# Функция для определения готовности к переезду
def get_ready_to_move(arg):
    if ('не готов к переезду' in arg) or ('не готова к переезду' in arg):
        return False
    elif 'хочу' in arg:
        return True
    else:
        return True

# Функция для определения готовности к командировкам
def get_ready_for_business_trips(arg):
    if 'командировка' in arg:
        if ('не готов к командировкам' in arg) or ('не готова к командировкам' in arg):
            return False
        else:
            return True
    else:
        return False

# Применяем функции к столбцу "Город, переезд, командировки"
df['Город'] = df['Город, переезд, командировки'].apply(get_city)
df['Готовность к переезду'] = df['Город, переезд, командировки'].apply(get_ready_to_move)
df['Готовность к командировкам'] = df['Город, переезд, командировки'].apply(get_ready_for_business_trips)

# Удаляем старый столбец
df.drop(columns=['Город, переезд, командировки'], inplace=True)

5. Рассмотрим поближе признаки **"Занятость"** и **"График"**. Сейчас признаки представляют собой набор категорий желаемой занятости (полная занятость, частичная занятость, проектная работа, волонтерство, стажировка) и желаемого графика работы (полный день, сменный график, гибкий график, удаленная работа, вахтовый метод).
На сайте hh.ru соискатель может указывать различные комбинации данных категорий, например:
* полная занятость, частичная занятость
* частичная занятость, проектная работа, волонтерство
* полный день, удаленная работа
* вахтовый метод, гибкий график, удаленная работа, полная занятость

Такой вариант признаков имеет множество различных комбинаций, а значит множество уникальных значений, что мешает анализу. Нужно это исправить!

Давайте создадим признаки-мигалки для каждой категории: если категория присутствует в списке желаемых соискателем, то в столбце на месте строки рассматриваемого соискателя ставится True, иначе - False.

Такой метод преобразования категориальных признаков называется One Hot Encoding и его схема представлена на рисунке ниже:
<img src=https://raw.githubusercontent.com/AndreyRysistov/DatasetsForPandas/main/ohe.jpg>
Выполните данное преобразование для признаков "Занятость" и "График", ответьте на контрольные вопросы, после чего удалите их из таблицы

In [ ]:
#ваш код здесь

# Преобразуем колонку "Занятость" в One Hot Encoding
employment_categories = ['полная занятость', 'частичная занятость', 'проектная работа', 'волонтерство', 'стажировка']
for category in employment_categories:
    df[category] = df['Занятость'].apply(lambda x: category in x if pd.notnull(x) else False)

# Преобразуем колонку "График" в One Hot Encoding
schedule_categories = ['полный день', 'сменный график', 'гибкий график', 'удалённая работа', 'вахтовый метод']
for category in schedule_categories:
    df[category] = df['График'].apply(lambda x: category in x if pd.notnull(x) else False)

# Удалим оригинальные столбцы "Занятость" и "График" (по необходимости)
df.drop(columns=['Занятость', 'График'], inplace=True)

# Проверим результат
print(df.head())

# Сохраним результат в новый CSV файл (опционально)
output_file_path = '/home/sexxlexx/Desktop/SKILL/dst.csv'
df.to_csv(output_file_path, index=False)

6. (2 балла) Наконец, мы добрались до самого главного и самого важного - признака заработной платы **"ЗП"**. 
В чем наша беда? В том, что помимо желаемой заработной платы соискатель указывает валюту, в которой он бы хотел ее получать, например:
* 30000 руб.
* 50000 грн.
* 550 USD

Нам бы хотелось видеть заработную плату в единой валюте, например, в рублях. Возникает вопрос, а где взять курс валют по отношению к рублю?

На самом деле язык Python имеет в арсенале огромное количество возможностей получения данной информации, от обращения к API Центробанка, до использования специальных библиотек, например pycbrf. Однако, это не тема нашего проекта.

Поэтому мы пойдем в лоб: обратимся к специальным интернет-ресурсам для получения данных о курсе в виде текстовых файлов. Например, MDF.RU, данный ресурс позволяет удобно экспортировать данные о курсах различных валют и акций за указанные периоды в виде csv файлов. Мы уже сделали выгрузку курсов валют, которые встречаются в наших данных за период с 29.12.2017 по 05.12.2019. Скачать ее вы можете **на платформе**

Создайте новый DataFrame из полученного файла. В полученной таблице нас будут интересовать столбцы:
* "currency" - наименование валюты в ISO кодировке,
* "date" - дата, 
* "proportion" - пропорция, 
* "close" - цена закрытия (последний зафиксированный курс валюты на указанный день).


Перед вами таблица соответствия наименований иностранных валют в наших данных и их общепринятых сокращений, которые представлены в нашем файле с курсами валют. Пропорция - это число, за сколько единиц валюты указан курс в таблице с курсами. Например, для казахстанского тенге курс на 20.08.2019 составляет 17.197 руб. за 100 тенге, тогда итоговый курс равен - 17.197 / 100 = 0.17197 руб за 1 тенге.
Воспользуйтесь этой информацией в ваших преобразованиях.

<img src=https://raw.githubusercontent.com/AndreyRysistov/DatasetsForPandas/main/table.jpg>


Осталось только понять, откуда брать дату, по которой определяется курс? А вот же она - в признаке **"Обновление резюме"**, в нем содержится дата и время, когда соискатель выложил текущий вариант своего резюме. Нас интересует только дата, по ней бы и будем сопоставлять курсы валют.

Теперь у нас есть вся необходимая информация для того, чтобы создать признак "ЗП (руб)" - заработная плата в рублях.

После ответа на контрольные вопросы удалите исходный столбец заработной платы "ЗП" и все промежуточные столбцы, если вы их создавали.

Итак, давайте обсудим возможный алгоритм преобразования: 
1. Перевести признак "Обновление резюме" из таблицы с резюме в формат datetime и достать из него дату. В тот же формат привести признак "date" из таблицы с валютами.
2. Выделить из столбца "ЗП" сумму желаемой заработной платы и наименование валюты, в которой она исчисляется. Наименование валюты перевести в стандарт ISO согласно с таблицей выше.
3. Присоединить к таблице с резюме таблицу с курсами по столбцам с датой и названием валюты (подумайте, какой тип объединения надо выбрать, чтобы в таблице с резюме сохранились данные о заработной плате, изначально представленной в рублях). Значение close для рубля заполнить единицей 1 (курс рубля самого к себе)
4. Умножить сумму желаемой заработной платы на присоединенный курс валюты (close) и разделить на пропорцию (обратите внимание на пропуски после объединения в этих столбцах), результат занести в новый столбец "ЗП (руб)".


In [ ]:
#ваш код здесь
#путь к курсу валют
exchange_file_path = '/home/sexlexx/Desktop/-Project-on-Data-Scientist-Courses/ExchangeRates.csv'

# Загрузка данных
df_exchange = pd.read_csv(exchange_file_path)

# Преобразование даты в обоих файлах
df['Обновление резюме'] = pd.to_datetime(df['Обновление резюме'], format='%d.%m.%Y %H:%M').dt.date
df_exchange['date'] = pd.to_datetime(df_exchange['date'], format='%d/%m/%y').dt.date

# Словарь соответствия валют
currency_mapping = {
    'руб.': ('RUB', 1),
    'грн.': ('UAH', 10),
    'USD': ('USD', 1),
    'EUR': ('EUR', 1),
    'бел.руб.': ('BYN', 1),
    'бел. руб.': ('BYN', 1),
    'KGS': ('KGS', 10),
    'сум': ('UZS', 10000),
    'AZN': ('AZN', 1),
    'KZT': ('KZT', 100)
}

# Функция для разбиения столбца "ЗП" на сумму и валюту
def parse_salary(salary):
    if pd.isna(salary):
        return pd.Series([None, None])
    try:
        amount, currency = salary.split(' ')
        return pd.Series([float(amount.replace('\u202f', '').replace(',', '')), currency])
    except:
        return pd.Series([None, None])

# Применяем функцию для создания новых столбцов "ЗП (сумма)" и "Валюта"
df[['ЗП (сумма)', 'Валюта']] = df['ЗП'].apply(parse_salary)

# Преобразуем валюту в ISO и добавляем пропорцию
df['ISO Валюта'] = df['Валюта'].map(lambda x: currency_mapping[x][0] if x in currency_mapping else None)
df['Пропорция'] = df['Валюта'].map(lambda x: currency_mapping[x][1] if x in currency_mapping else None)

# Добавляем курс валют
df = pd.merge(
    df,
    df_exchange,
    how='left',
    left_on=['Обновление резюме', 'ISO Валюта'],
    right_on=['date', 'currency']
)

# Заполняем курс рубля (RUB) как 1
df['close'] = df['close'].fillna(1)

# Рассчитываем зарплату в рублях
df['ЗП (руб)'] = (df['ЗП (сумма)'] * df['close'] / df['Пропорция']).round(2)

# Диагностика: строки, не попавшие в расчёт
unmatched = df[df['ЗП (руб)'].isna()]
if not unmatched.empty:
    print("Не удалось рассчитать заработную плату для следующих записей:")
    print(unmatched[['Обновление резюме', 'ЗП', 'ISO Валюта', 'close', 'Пропорция']])

# Удаление ненужных столбцов
columns_to_drop = ['ЗП', 'ЗП (сумма)', 'Валюта', 'ISO Валюта', 'Пропорция', 'date', 'currency', 'close']
df.drop(columns=columns_to_drop, inplace=True)

columns_to_remove = ['per', 'time', 'vol', 'proportion']
df.drop(columns=[col for col in columns_to_remove if col in df.columns], inplace=True)

# Контрольный вопрос: медианная зарплата
median_salary_rub = int(df['ЗП (руб)'].median() // 1000)  # Приводим к тысячам и округляем до целого
print(f"Медианная заработная плата соискателей в рублях (в тысячах): {median_salary_rub}")


# Исследование зависимостей в данных

1. Постройте распределение признака **"Возраст"**. Опишите распределение, отвечая на следующие вопросы: чему равна мода распределения, каковы предельные значения признака, в каком примерном интервале находится возраст большинства соискателей? Есть ли аномалии для признака возраста, какие значения вы бы причислили к их числу?
*Совет: постройте гистограмму и коробчатую диаграмму рядом.*

In [11]:
# ваш код здесь
# Проверяем, что столбец "Возраст" существует
if 'Возраст' not in df.columns:
    raise KeyError("Столбец 'Возраст' отсутствует в данных!")

# Статистические параметры
mode_age = df['Возраст'].mode()[0]  # Мода
min_age = df['Возраст'].min()  # Минимум
max_age = df['Возраст'].max()  # Максимум

# Вывод статистики
print(f"Мода распределения: {mode_age}")
print(f"Минимальный возраст: {min_age}")
print(f"Максимальный возраст: {max_age}")

# Построение гистограммы
plt.figure(figsize=(20, 8))

# Гистограмма
plt.subplot(1, 2, 1)
bins = range(min_age, max_age + 2)  # Интервалы для каждого возраста
counts, bins, bars = plt.hist(df['Возраст'], bins=bins, color='skyblue', edgecolor='black', rwidth=0.8)
plt.title('Распределение возраста (гистограмма)', fontsize=16)
plt.xlabel('Возраст', fontsize=14)
plt.ylabel('Частота', fontsize=14)
plt.xticks(bins[:-1], rotation=45)  # Отображаем каждое значение возраста

# Отображение количества замеров на верхушках столбцов
for count, bin_edge in zip(counts, bins[:-1]):
    plt.text(bin_edge + 0.5, count, str(int(count)),
             ha='center', va='bottom', fontsize=10, color='black')

# Коробчатая диаграмма
plt.subplot(1, 2, 2)
plt.boxplot(df['Возраст'], vert=False, patch_artist=True, boxprops=dict(facecolor='skyblue'))
plt.title('Распределение возраста (коробчатая диаграмма)', fontsize=16)
plt.xlabel('Возраст', fontsize=14)

# Отображение графиков
plt.tight_layout()

1. Мода распределения (30 лет):
Большинство соискателей имеют возраст 30 лет, что является самым частотным значением.
Это значение указывает на то, что большая часть выборки состоит из специалистов среднего возраста, которые, вероятно, уже имеют некоторый опыт работы.
2. Минимальный и максимальный возраст:
Минимальный возраст: 14 лет. Это значение может быть ошибкой или связано с редкими случаями юных соискателей.
Максимальный возраст: 100 лет. Это значение также может быть ошибкой, поскольку в реальной жизни соискатели такого возраста встречаются крайне редко.
3. Гистограмма:
Распределение имеет симметричный характер с правосторонним "хвостом".
Большинство значений сосредоточено в диапазоне 20–40 лет, что отражает возрастную группу активных работников.
После 40 лет частота резко снижается, что может быть связано с уменьшением числа соискателей в старшей возрастной группе.
4. Коробчатая диаграмма:
Основная масса значений находится в диапазоне 20–40 лет, что соответствует медианному возрасту трудоспособного населения.
Выбросы: Значения выше 60 лет и минимальное значение 14 лет являются выбросами. Это может быть следствием ошибок в данных или редких случаев.
5. Выводы и рекомендации:
Основная возрастная группа соискателей: 20–40 лет.
Выбросы (меньше 18 лет и больше 60 лет) стоит рассмотреть подробнее:
Убедиться, что эти данные корректны.
При необходимости исключить из анализа значения меньше 18 лет (недопустимый для работы возраст) и значения старше 60–65 лет (пенсионный возраст в большинстве стран).
Данные можно сгруппировать в диапазоны (например, 18–25, 26–35, 36–45 и т. д.) для анализа возрастных групп и их предпочтений.
Графики четко демонстрируют закономерности и выделяют аномальные значения, что помогает сделать более точные выводы о возрастной структуре соискателей.

2. Постройте распределение признака **"Опыт работы (месяц)"**. Опишите данное распределение, отвечая на следующие вопросы: чему равна мода распределения, каковы предельные значения признака, в каком примерном интервале находится опыт работы большинства соискателей? Есть ли аномалии для признака опыта работы, какие значения вы бы причислили к их числу?
*Совет: постройте гистограмму и коробчатую диаграмму рядом.*

In [ ]:
# ваш код здесь
# Убедимся, что столбец «Опыт работы (месяц)» существует
if 'Опыт работы (месяц)' not in df.columns:
    raise KeyError("Столбец 'Опыт работы (месяц)' отсутствует в данных!")

# Удалим строки с отсутствующим значением
df = df.dropna(subset=['Опыт работы (месяц)'])

# Статистические параметры
mode_experience = df['Опыт работы (месяц)'].mode()[0]  # Мода
min_experience = df['Опыт работы (месяц)'].min()  # Минимум
max_experience = df['Опыт работы (месяц)'].max()  # Максимум

# Определим порог для аномалий
threshold_high = 500  # Аномалии выше 500 месяцев
anomalies = df[df['Опыт работы (месяц)'] > threshold_high]['Опыт работы (месяц)']

# Вывод статистики
print(f"Мода распределения: {mode_experience}")
print(f"Минимальный опыт работы: {min_experience} месяцев")
print(f"Максимальный опыт работы: {max_experience} месяцев")
print(f"Аномалии (выше {threshold_high} месяцев):")
print(anomalies.unique())

# Построение графиков
plt.figure(figsize=(20, 8))

# Гистограмма
plt.subplot(1, 2, 1)
plt.hist(df['Опыт работы (месяц)'], bins=50, color='orange', edgecolor='black')
plt.title('Распределение опыта работы (гистограмма)', fontsize=16)
plt.xlabel('Опыт работы (месяц)', fontsize=14)
plt.ylabel('Частота', fontsize=14)

# Коробчатая диаграмма
plt.subplot(1, 2, 2)
plt.boxplot(df['Опыт работы (месяц)'], vert=False, patch_artist=True, boxprops=dict(facecolor='orange'))
plt.title('Распределение опыта работы (коробчатая диаграмма)', fontsize=16)
plt.xlabel('Опыт работы (месяц)', fontsize=14)

# Отображение графиков
plt.tight_layout()
plt.show()

1. Мода распределения (81 месяц):
Большинство соискателей имеют опыт работы около 81 месяца (примерно 6,75 лет), что является самым частотным значением.
Это значение говорит о том, что значительная часть выборки представляет специалистов с опытом в диапазоне среднего уровня.
2. Минимальное и максимальное значения:
Минимальное значение: 1 месяц. Это указывает на наличие начинающих специалистов, которые только начинают карьеру.
Максимальное значение: 1188 месяцев (99 лет). Это явная аномалия, поскольку она значительно превышает реальный возможный трудовой стаж.
3. Аномалии:
Значения выше 500 месяцев (около 41,7 лет) можно считать аномальными. Они не соответствуют реалистичному сценарию для большинства профессионалов.
Примеры аномальных значений: 507, 510, 663, 1188 месяцев. Эти данные требуют проверки или обработки (например, удаления или замены).
4. Распределение данных (гистограмма):
Распределение имеет правосторонний асимметричный характер:
Большинство данных сосредоточено в диапазоне 0-200 месяцев (0-16,7 лет).
Значения выше 200 месяцев резко снижаются по частоте, что естественно для трудового стажа.
5. Коробчатая диаграмма:
Выявлены выбросы, которые выходят за пределы верхнего уса диаграммы.
Верхний ус достигает значений около 200-250 месяцев, что соответствует стажу в 16-20 лет.
Аномалии выше 500 месяцев (выбросы) явно выделяются на диаграмме и требуют проверки.
6. Выводы и рекомендации:
Основная масса соискателей имеет опыт работы от 1 до 200 месяцев.
Данные с опытом выше 500 месяцев следует:
Либо удалить как аномалии.
Либо заменить, если они могут быть результатом ошибок в данных (например, некорректное форматирование).
Данные можно сгруппировать в диапазоны для упрощения анализа (например, 0-50, 51-100, 101-150 месяцев). Это позволит лучше визуализировать ключевые тренды.
Графики подтверждают правомерность вышеприведенных выводов и помогают визуально выделить ключевые особенности распределения.

3. Постройте распределение признака **"ЗП (руб)"**. Опишите данное распределение, отвечая на следующие вопросы: каковы предельные значения признака, в каком примерном интервале находится заработная плата большинства соискателей? Есть ли аномалии для признака возраста? Обратите внимание на гигантские размеры желаемой заработной платы.
*Совет: постройте гистограмму и коробчатую диаграмму рядом.*


In [ ]:
# ваш код здесь
# Проверяем, что столбец "ЗП (руб)" существует
if 'ЗП (руб)' not in df.columns:
    raise KeyError("Столбец 'ЗП (руб)' отсутствует в данных!")

# Удаляем строки с отсутствующими значениями
df = df.dropna(subset=['ЗП (руб)'])

# Статистические параметры
mode_salary = df['ЗП (руб)'].mode()[0]  # Мода
min_salary = df['ЗП (руб)'].min()  # Минимум
max_salary = df['ЗП (руб)'].max()  # Максимум

# Определим порог для аномалий
threshold_high = 500_000  # Аномалии выше 500,000 рублей
anomalies = df[df['ЗП (руб)'] > threshold_high]['ЗП (руб)']

# Вывод статистики
print(f"Мода распределения: {mode_salary} руб.")
print(f"Минимальная заработная плата: {min_salary} руб.")
print(f"Максимальная заработная плата: {max_salary} руб.")
print(f"Аномалии (выше {threshold_high} руб.):")
print(anomalies.unique())

# Построение графиков
plt.figure(figsize=(20, 8))

# Гистограмма
plt.subplot(1, 2, 1)
plt.hist(df['ЗП (руб)'], bins=50, color='green', edgecolor='black')
plt.title('Распределение заработной платы (гистограмма)', fontsize=16)
plt.xlabel('Заработная плата (руб)', fontsize=14)
plt.ylabel('Частота', fontsize=14)

# Коробчатая диаграмма
plt.subplot(1, 2, 2)
plt.boxplot(df['ЗП (руб)'], vert=False, patch_artist=True, boxprops=dict(facecolor='green'))
plt.title('Распределение заработной платы (коробчатая диаграмма)', fontsize=16)
plt.xlabel('Заработная плата (руб)', fontsize=14)

# Отображение графиков
plt.tight_layout()

1. Мода распределения (50000 руб.):
Наиболее частое значение заработной платы составляет 50,000 рублей. Это типичная зарплата, которая встречается у большинства соискателей.
2. Минимальная заработная плата (1 руб.):
Значение в 1 рубль, вероятно, является ошибкой или некорректным вводом данных. Такие записи можно рассматривать как аномалии.
3. Максимальная заработная плата (24,304,876 руб.):
Очень высокая зарплата, которая явно выделяется среди остальных значений. Это аномалия, так как такие значения значительно превышают нормальные зарплаты.
4. Аномалии (выше 500,000 руб.):
Значения выше 500,000 руб. включают:
Зарплаты около 600,000–700,000 руб. могут встречаться у топ-менеджеров.
Зарплаты в миллионах (например, 24,304,876 руб. или 7,675,224 руб.) явно являются либо ошибками ввода, либо очень редкими исключениями.
5. Распределение на гистограмме:
Большая часть значений сосредоточена в диапазоне до 100,000 руб., что свидетельствует о том, что большинство соискателей запрашивают относительно стандартные зарплаты.
Практически не видно данных за пределами 500,000 руб. из-за большого дисбаланса в распределении.
6. Коробчатая диаграмма:
Медиана и межквартильный диапазон показывают, что основная масса заработных плат лежит в пределах до 100,000 руб.
Большое количество выбросов, начиная от 500,000 руб., сильно смещает диапазон распределения.
7. Рекомендации:
Рассмотреть удаление или обработку записей с зарплатами ниже 10 руб. и выше 500,000 руб., так как это либо ошибки, либо редкие исключения.
Уточнить критерии определения зарплат выше 500,000 руб., чтобы учитывать профессиональные роли, в которых такие значения являются оправданными.
Для более детального анализа можно построить графики без учета аномалий. Это позволит лучше понять основные тенденции заработной платы.

4. Постройте диаграмму, которая показывает зависимость **медианной** желаемой заработной платы (**"ЗП (руб)"**) от уровня образования (**"Образование"**). Используйте для диаграммы данные о резюме, где желаемая заработная плата меньше 1 млн рублей.
*Сделайте выводы по представленной диаграмме: для каких уровней образования наблюдаются наибольшие и наименьшие уровни желаемой заработной платы? Как вы считаете, важен ли признак уровня образования при прогнозировании заработной платы?*

In [ ]:
# ваш код здесь
# Фильтруем данные, оставляем только записи с заработной платой меньше 1 миллиона рублей
filtered_df = df[df['ЗП (руб)'] < 1_000_000]

# Рассчитываем медианную зарплату по уровням образования
median_salary_by_education = filtered_df.groupby('Образование')['ЗП (руб)'].median().sort_values()

# Построение диаграммы
plt.figure(figsize=(10, 6))
median_salary_by_education.plot(kind='bar', color='skyblue', edgecolor='black')
plt.title('Зависимость медианной желаемой заработной платы от уровня образования', fontsize=14)
plt.xlabel('Уровень образования', fontsize=12)
plt.ylabel('Медианная желаемая заработная плата (руб)', fontsize=12)
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()

# Сохранение графика (опционально)
plt.savefig('/home/sexxlexx/Desktop/SKILL/median_salary_by_education.png')


1. Наибольшая медианная заработная плата:
Соискатели с высшим образованием имеют наибольшую медианную заработную плату среди всех уровней образования.
Это свидетельствует о положительном влиянии высшего образования на уровень желаемой заработной платы.
2. Наименьшая медианная заработная плата:
Соискатели со средним образованием запрашивают наименьшую медианную заработную плату.
Это может быть связано с тем, что соискатели с таким уровнем образования чаще занимают низкооплачиваемые должности.
3. Различия между уровнями образования:
Медианная зарплата растет с увеличением уровня образования:
Среднее образование < Среднее специальное < Неоконченное высшее < Высшее.
Это подтверждает, что уровень образования является значимым признаком при прогнозировании заработной платы.
4. Практическая значимость уровня образования:
Уровень образования оказывает значительное влияние на желаемую заработную плату. Это важно учитывать в моделях прогнозирования заработной платы или при планировании профессионального роста.
В целом, диаграмма демонстрирует очевидную корреляцию между уровнем образования и уровнем медианной заработной платы.

5. Постройте диаграмму, которая показывает распределение желаемой заработной платы (**"ЗП (руб)"**) в зависимости от города (**"Город"**). Используйте для диаграммы данные о резюме, где желая заработная плата меньше 1 млн рублей.
*Сделайте выводы по полученной диаграмме: как соотносятся медианные уровни желаемой заработной платы и их размах в городах? Как вы считаете, важен ли признак города при прогнозировании заработной платы?*

In [ ]:
# ваш код здесь
# Удалим данные с зарплатой выше 1 миллиона рублей
df_filtered = df[df['ЗП (руб)'] < 1000000]

# Построим диаграмму распределения желаемой зарплаты по городам
plt.figure(figsize=(14, 8))
sns.boxplot(data=df_filtered, x='Город', y='ЗП (руб)', hue='Город', palette='coolwarm', dodge=False, legend=False)
plt.title('Распределение желаемой заработной платы по городам', fontsize=16)
plt.xlabel('Город', fontsize=14)
plt.ylabel('Заработная плата (руб)', fontsize=14)
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()

# Сохранение диаграммы
plt.savefig('/home/sexxlexx/Desktop/SKILL/city_salary_distribution.png')

1. Медианные уровни желаемой заработной платы:
Москва демонстрирует наибольшую медианную заработную плату среди всех категорий городов. Это ожидаемо, учитывая статус столицы и высокий уровень жизни.
Санкт-Петербург также показывает относительно высокий медианный уровень заработной платы, но он ниже, чем в Москве.
Города-миллионники занимают среднее положение, их медианная заработная плата ниже, чем в Москве и Санкт-Петербурге.
Другие города демонстрируют наименьшую медианную заработную плату, что может быть связано с более низкими запросами в регионах.
2. Размах заработной платы:
В Москве размах заработной платы наиболее заметен, включая большое количество выбросов, которые значительно превышают медиану.
Города-миллионники и Санкт-Петербург имеют схожий размах заработной платы, но он меньше, чем в Москве.
В других городах размах минимален, что говорит о более равномерных запросах.
3. Важность признака города:
Признак города играет важную роль в прогнозировании заработной платы, поскольку существует явная корреляция между местоположением и уровнем желаемой заработной платы. Соискатели из Москвы и Санкт-Петербурга запрашивают значительно больше, чем из других категорий городов.
Признак города является критически важным для прогнозирования заработной платы, так как различия между категориями городов (особенно между Москвой и другими регионами) значительны. Москва и Санкт-Петербург выделяются как города с высокими зарплатными ожиданиями и большим размахом значений.

6. Постройте **многоуровневую столбчатую диаграмму**, которая показывает зависимость медианной заработной платы (**"ЗП (руб)"**) от признаков **"Готовность к переезду"** и **"Готовность к командировкам"**. Проанализируйте график, сравнив уровень заработной платы в категориях.

In [ ]:
# ваш код здесь
# Отфильтруем заработные платы менее 1 млн рублей
df_filtered = df[df['ЗП (руб)'] < 1_000_000]

# Группировка по категориям
median_salary = df_filtered.groupby(['Готовность к переезду', 'Готовность к командировкам'])['ЗП (руб)'].median().reset_index()

# Добавим столбец для подписи категорий
median_salary['Категория'] = median_salary.apply(
    lambda x: f"Переезд: {'Да' if x['Готовность к переезду'] else 'Нет'}, Командировки: {'Да' if x['Готовность к командировкам'] else 'Нет'}",
    axis=1
)

# Построение столбчатой диаграммы
plt.figure(figsize=(10, 6))
sns.barplot(
    data=median_salary,
    x='Категория',
    y='ЗП (руб)',
    palette='coolwarm'
)
plt.title('Медианная заработная плата в зависимости от готовности к переезду и командировкам', fontsize=14)
plt.xlabel('Категория', fontsize=12)
plt.ylabel('Медианная заработная плата (руб)', fontsize=12)
plt.xticks(rotation=45, fontsize=10, ha='right')
plt.tight_layout()

# Сохранение графика
plt.savefig('/home/sexxlexx/Desktop/SKILL/median_salary_by_mobility.png')

In [ ]:
1. Медианные заработные платы:
Соискатели, не готовые ни к переезду, ни к командировкам, имеют самую низкую медианную заработную плату (около 40 тысяч рублей).
Соискатели, готовые к командировкам, но не готовые к переезду, имеют более высокую медианную заработную плату (около 60 тысяч рублей).
Соискатели, готовые к переезду, но не готовые к командировкам, имеют медианную заработную плату примерно на уровне 50 тысяч рублей.
Наивысшая медианная заработная плата (около 65 тысяч рублей) зафиксирована у тех, кто готов как к переезду, так и к командировкам.
2. Зависимость от мобильности:
Готовность к командировкам и переезду явно повышает уровень медианной заработной платы. Это может быть связано с тем, что такие кандидаты считаются более гибкими и открытыми для предложений, связанных с высокооплачиваемыми должностями.
3. Важность признака:
Признаки "Готовность к переезду" и "Готовность к командировкам" важны для прогнозирования заработной платы, поскольку наличие этих готовностей приводит к заметным различиям в уровнях медианной зарплаты.
4. Общий тренд:
Наблюдается положительная корреляция между готовностью к мобильности (переезд и командировки) и желаемой заработной платой.
Таким образом, можно сделать вывод, что наибольший уровень медианной заработной платы наблюдается у соискателей, готовых и к переезду, и к командировкам, что указывает на высокую востребованность таких кандидатов.

7. Постройте сводную таблицу, иллюстрирующую зависимость **медианной** желаемой заработной платы от возраста (**"Возраст"**) и образования (**"Образование"**). На полученной сводной таблице постройте **тепловую карту**. Проанализируйте тепловую карту, сравнив показатели внутри групп.

In [ ]:
# ваш код здесь
# Фильтруем данные: заработная плата < 1 миллиона рублей
df_filtered = df[df['ЗП (руб)'] < 1_000_000]

# Построение сводной таблицы
pivot_table = df_filtered.pivot_table(
    index='Возраст', 
    columns='Образование', 
    values='ЗП (руб)', 
    aggfunc='median'
)

# Построение тепловой карты
plt.figure(figsize=(12, 8))
sns.heatmap(pivot_table, cmap='coolwarm', annot=False, fmt=".0f", cbar_kws={'label': 'Медианная ЗП (руб)'})
plt.title('Тепловая карта медианной заработной платы по возрасту и образованию', fontsize=16)
plt.xlabel('Образование', fontsize=12)
plt.ylabel('Возраст', fontsize=12)
plt.tight_layout()

# Сохранение графика (опционально)
plt.savefig('/home/sexxlexx/Desktop/SKILL/heatmap_salary_by_age_education.png')

1. Интенсивность роста заработной платы:
На тепловой карте видны ярко выраженные различия между уровнями образования.
Высшее образование демонстрирует наибольший рост медианной заработной платы с возрастом. Это подтверждается более тёплыми тонами (красными), особенно в возрастной категории от 30 до 50 лет.
Неоконченное высшее образование показывает также значительное повышение зарплат, но интенсивность роста ниже, чем у высшего образования.
Среднее образование и среднее специальное образование имеют более низкую медианную зарплату, а рост с возрастом менее выражен.
2. Наибольшая медианная зарплата:
Самые высокие значения медианной заработной платы наблюдаются у людей с высшим образованием в возрасте около 40–50 лет.
3. Медленный карьерный рост:
Для соискателей с средним специальным и средним образованием рост заработной платы практически не заметен, и медианные зарплаты остаются низкими.
4. Зависимость образования и карьеры:
Видно, что уровень образования является критически важным фактором для прогнозирования заработной платы. Высшее образование предоставляет значительно больше возможностей для карьерного роста и увеличения доходов.

8. Постройте **диаграмму рассеяния**, показывающую зависимость опыта работы (**"Опыт работы (месяц)"**) от возраста (**"Возраст"**). Опыт работы переведите из месяцев в года, чтобы признаки были в едином масштабе. Постройте на графике дополнительно прямую, проходящую через точки (0, 0) и (100, 100). Данная прямая соответствует значениям, когда опыт работы равен возрасту человека. Точки, лежащие на этой прямой и выше нее - аномалии в наших данных (опыт работы больше либо равен возрасту соискателя)

In [ ]:
# ваш код здесь
# Убедимся, что необходимые столбцы существуют
if 'Опыт работы (месяц)' not in df.columns or 'Возраст' not in df.columns:
    raise KeyError("Столбцы 'Опыт работы (месяц)' и/или 'Возраст' отсутствуют в данных!")

# Удалим строки с отсутствующими значениями
df = df.dropna(subset=['Опыт работы (месяц)', 'Возраст'])

# Преобразуем опыт работы из месяцев в годы
df['Опыт работы (год)'] = df['Опыт работы (месяц)'] / 12

# Построим диаграмму рассеяния
plt.figure(figsize=(10, 6))
plt.scatter(df['Возраст'], df['Опыт работы (год)'], alpha=0.6, label='Соискатели')

# Добавим прямую, где опыт работы равен возрасту
plt.plot([0, 100], [0, 100], color='red', linestyle='--', label='Опыт работы = Возраст')

# Подписи графика
plt.title('Зависимость опыта работы от возраста', fontsize=14)
plt.xlabel('Возраст (годы)', fontsize=12)
plt.ylabel('Опыт работы (годы)', fontsize=12)
plt.legend()

# Сохраняем график
plt.savefig('/home/sexxlexx/Desktop/SKILL/scatterplot_experience_vs_age.png')
plt.show()

# Определим точки выше прямой
anomalies = df[df['Опыт работы (год)'] > df['Возраст']]

# Вывод количества аномалий
print(f"Количество точек выше прямой: {len(anomalies)}")

# Сохранение таблицы с аномалиями (опционально)
anomalies.to_csv('/home/sexxlexx/Desktop/SKILL/anomalies.csv', index=False)

1. Основное распределение:
Большинство точек на графике находится ниже линии, соответствующей равенству возраста и опыта работы. Это ожидаемо, так как опыт работы не может превышать возраст за вычетом времени обучения и других факторов.
2. Аномалии:
На графике есть точки, которые лежат на линии или выше неё. Это означает, что в данных есть кандидаты, чей опыт работы равен или превышает их возраст. Такие случаи являются аномалиями, так как это не соответствует реальности.
3. Возрастные группы:
Для возрастов от 20 до 60 лет наблюдается плотное распределение точек с опытом работы в диапазоне 0–30 лет. Это свидетельствует о большей активности на рынке труда в этой возрастной группе.
У людей младше 20 лет опыт работы практически отсутствует (логично, так как это период учебы).
У людей старше 60 лет есть точки с высоким опытом работы, что может отражать либо реальный долгий трудовой стаж, либо ошибки в данных.
4. Линия равенства:
Красная пунктирная линия чётко демонстрирует границу, выше которой опыт работы превышает возраст. Количество точек выше линии относительно невелико, но их наличие указывает на возможные проблемы в данных, такие как ошибки в заполнении анкет или некорректная интерпретация данных.
5. Практическое значение:
Этот анализ помогает выявить потенциальные ошибки в данных и аномалии. Такие случаи необходимо либо уточнить, либо исключить из модели, чтобы избежать искажения результатов.
6. Рекомендация:
Провести дополнительный анализ записей, где опыт работы больше или равен возрасту. Возможно, потребуется связаться с авторами резюме для уточнения данных.

**Дополнительные баллы**

Для получения 2 дополнительных баллов по разведывательному анализу постройте еще два любых содержательных графика или диаграммы, которые помогут проиллюстрировать влияние признаков/взаимосвязь между признаками/распределения признаков. Приведите выводы по ним. Желательно, чтобы в анализе участвовали признаки, которые мы создавали ранее в разделе "Преобразование данных".


In [ ]:
# Преобразуем опыт работы в годы
df['Опыт работы (годы)'] = df['Опыт работы (месяц)'] / 12

# Фильтруем данные
df_filtered = df[df['ЗП (руб)'] < 1_000_000]

# Группируем данные и рассчитываем медианную зарплату
median_salary_by_experience = df_filtered.groupby('Опыт работы (годы)')['ЗП (руб)'].median()

# Построение графика
plt.figure(figsize=(10, 6))
plt.plot(median_salary_by_experience.index, median_salary_by_experience.values, marker='o', color='blue')
plt.title('Медианная заработная плата в зависимости от опыта работы', fontsize=16)
plt.xlabel('Опыт работы (годы)', fontsize=14)
plt.ylabel('Медианная заработная плата (руб)', fontsize=14)
plt.grid(True)
plt.tight_layout()
plt.savefig('/home/sexxlexx/Desktop/SKILL/median_salary_by_experience.png')

# Отбираем данные с зарплатой ниже 1 миллиона рублей
df_filtered = df[df['ЗП (руб)'] < 1_000_000]

# Определяем графики работы
schedule_categories = ['полный день', 'сменный график', 'гибкий график', 'удалённая работа', 'вахтовый метод']

# Рассчитываем медианную зарплату по каждому графику
median_salary_by_schedule = {category: df_filtered[df_filtered[category] == True]['ЗП (руб)'].median() 
                             for category in schedule_categories}

# Преобразуем словарь в два списка: категории и медианные значения
categories = list(median_salary_by_schedule.keys())
median_salaries = list(median_salary_by_schedule.values())

# Построение графика
plt.figure(figsize=(10, 6))
plt.bar(categories, median_salaries, color='orange', edgecolor='black')
plt.title('Медианная заработная плата в зависимости от графика работы', fontsize=16)
plt.xlabel('График работы', fontsize=14)
plt.ylabel('Медианная заработная плата (руб)', fontsize=14)
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig('/home/sexxlexx/Desktop/SKILL/median_salary_by_schedule.png')

# Создаем возрастные группы
bins = [0, 25, 35, 45, 55, 65, 100]
labels = ['<25', '25-35', '35-45', '45-55', '55-65', '65+']
df_filtered['Возрастная группа'] = pd.cut(df_filtered['Возраст'], bins=bins, labels=labels, right=False)

# Рассчитываем медианную зарплату для каждой группы
median_salary_by_age_group = df_filtered.groupby('Возрастная группа')['ЗП (руб)'].median()

# Построение графика
plt.figure(figsize=(10, 6))
median_salary_by_age_group.plot(kind='bar', color='purple', edgecolor='black')
plt.title('Медианная заработная плата по возрастным группам', fontsize=16)
plt.xlabel('Возрастная группа', fontsize=14)
plt.ylabel('Медианная заработная плата (руб)', fontsize=14)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig('/home/sexxlexx/Desktop/SKILL/median_salary_by_age_group.png')

# Создаем новую категорию для комбинированного анализа
df_filtered['Образование + Переезд'] = df_filtered['Образование'] + ' | Переезд: ' + df_filtered['Готовность к переезду'].map({True: 'Да', False: 'Нет'})

# Рассчитываем медианную зарплату для каждой комбинации
median_salary_by_combination = df_filtered.groupby('Образование + Переезд')['ЗП (руб)'].median().sort_values()

# Построение графика
plt.figure(figsize=(12, 8))
median_salary_by_combination.plot(kind='barh', color='teal', edgecolor='black')
plt.title('Медианная заработная плата в зависимости от уровня образования и готовности к переезду', fontsize=16)
plt.xlabel('Медианная заработная плата (руб)', fontsize=14)
plt.ylabel('Категория (Образование + Переезд)', fontsize=14)
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig('/home/sexxlexx/Desktop/SKILL/median_salary_by_education_mobility.png')

1. Медианная заработная плата по возрастным группам:
Самая низкая медианная заработная плата наблюдается у соискателей младше 25 лет.
Наивысший уровень медианной заработной платы отмечается у соискателей старше 65 лет. Это может быть связано с опытом работы и высокой квалификацией в их возрасте.
В возрасте 25–45 лет наблюдается рост медианной зарплаты, который замедляется после 45 лет.
2. Медианная заработная плата в зависимости от уровня образования и готовности к переезду:
Соискатели с высшим образованием имеют наибольшую медианную заработную плату, особенно если они готовы к переезду.
Соискатели со средним специальным образованием имеют наименьшую медианную зарплату, независимо от готовности к переезду.
Готовность к переезду повышает медианную зарплату практически для всех категорий образования, что может быть связано с доступом к более высоким позициям в других регионах.
3. Медианная заработная плата в зависимости от опыта работы (в годах):
Наблюдается плавный рост медианной заработной платы по мере увеличения опыта работы, особенно в диапазоне от 0 до 20 лет.
После 20 лет опыта рост становится менее выраженным, а данные для опытов более 40 лет выглядят аномальными, вероятно, из-за небольшого количества данных.
4. Медианная заработная плата в зависимости от графика работы:
Наивысшая медианная заработная плата отмечается для работников с графиком "полный день" и "гибкий график".
Наименьшая медианная зарплата наблюдается для работников с "вахтовым методом" и "сменным графиком".
Это может указывать на то, что тип графика работы оказывает влияние на уровень заработной платы, при этом полный день и гибкий график чаще ассоциируются с более высокими должностями.
5. Общие выводы:
Уровень образования и готовность к переезду оказывают значительное влияние на уровень медианной заработной платы.
График работы и возраст также являются важными факторами при анализе заработной платы.
Необходимо учитывать аномалии, особенно для высоких опытов работы и возрастных групп старше 65 лет.

# Очистка данных

1. Начнем с дубликатов в наших данных. Найдите **полные дубликаты** в таблице с резюме и удалите их. 

In [ ]:
# ваш код здесь
file_path = '/home/sexxlexx/Desktop/SKILL/dst.csv'

# Чтение данных с указанием разделителя ";"
df = pd.read_csv(file_path, sep=",")

# Определение количества дубликатов
duplicates_count = df.duplicated().sum()
print(f"Количество полных дубликатов: {duplicates_count}")

# Удаление дубликатов
df_cleaned = df.drop_duplicates()

# Сохранение очищенных данных
output_file_path = '/home/sexxlexx/Desktop/SKILL/dst_cleaned.csv'
df_cleaned.to_csv(output_file_path, index=False)
print(f"Данные без дубликатов сохранены в файл: {output_file_path}")

2. Займемся пропусками. Выведите информацию **о числе пропусков** в столбцах. 

In [ ]:
# ваш код здесь
file_path = '/home/sexxlexx/Desktop/SKILL/dst_cleaned.csv'

# Чтение очищенных данных
df = pd.read_csv(file_path, sep=",")

# Подсчёт пропусков в каждом столбце
missing_values = df.isnull().sum()

# Вывод пропусков
print("Число пропусков в каждом столбце:")
print(missing_values)

3. Итак, у нас есть пропуски в 3ех столбцах: **"Опыт работы (месяц)"**, **"Последнее/нынешнее место работы"**, **"Последняя/нынешняя должность"**. Поступим следующим образом: удалите строки, где есть пропуск в столбцах с местом работы и должностью. Пропуски в столбце с опытом работы заполните **медианным** значением.

In [ ]:
# ваш код здесь
file_path = '/home/sexxlexx/Desktop/SKILL/dst_cleaned.csv'

# Чтение данных
df = pd.read_csv(file_path, sep=",")

# Удаляем строки, где есть пропуски в столбцах "Последнее/нынешнее место работы" и "Последняя/нынешняя должность"
df = df.dropna(subset=['Последнее/нынешнее место работы', 'Последняя/нынешняя должность'])

# Заполняем пропуски в столбце "Опыт работы (месяц)" медианным значением
median_experience = df['Опыт работы (месяц)'].median()
df['Опыт работы (месяц)'].fillna(median_experience, inplace=True)

# Сохранение очищенных данных в новый файл (опционально)
output_file_path = '/home/sexxlexx/Desktop/SKILL/dst_cleaned_no_missing.csv'
df.to_csv(output_file_path, index=False)

4. Мы добрались до ликвидации выбросов. Сначала очистим данные вручную. Удалите резюме, в которых указана заработная плата либо выше 1 млн. рублей, либо ниже 1 тыс. рублей.

In [ ]:
# ваш код здесь
file_path = '/home/sexxlexx/Desktop/SKILL/dst_cleaned.csv'

# Чтение данных
df = pd.read_csv(file_path, sep=",")

outliers = df[(df['ЗП (руб)'] < 1000) | (df['ЗП (руб)'] > 1_000_000)]

# Подсчёт выбросов
num_outliers = len(outliers)
print(f"Число выбросов: {num_outliers}")

# Удаление выбросов из данных
df = df[(df['ЗП (руб)'] >= 1000) & (df['ЗП (руб)'] <= 1_000_000)]

# Проверка новой размерности таблицы
print(f"Новая размерность таблицы: {df.shape}")

5. В процессе разведывательного анализа мы обнаружили резюме, в которых **опыт работы в годах превышал возраст соискателя**. Найдите такие резюме и удалите их из данных


In [12]:
# ваш код здесь
# Условие для выбросов: опыт работы (в годах) > возраст
outliers_experience = df[df['Опыт работы (месяц)'] / 12 > df['Возраст']]

# Подсчёт выбросов
num_outliers_experience = len(outliers_experience)
print(f"Число выбросов: {num_outliers_experience}")

# Удаление выбросов из данных
df = df[df['Опыт работы (месяц)'] / 12 <= df['Возраст']]

# Проверка новой размерности таблицы
print(f"Новая размерность таблицы: {df.shape}")

6. В результате анализа мы обнаружили потенциальные выбросы в признаке **"Возраст"**. Это оказались резюме людей чересчур преклонного возраста для поиска работы. Попробуйте построить распределение признака в **логарифмическом масштабе**. Добавьте к графику линии, отображающие **среднее и границы интервала метода трех сигм**. Напомним, сделать это можно с помощью метода axvline. Например, для построение линии среднего будет иметь вид:

`histplot.axvline(log_age.mean(), color='k', lw=2)`

В какую сторону асимметрично логарифмическое распределение? Напишите об этом в комментарии к графику.
Найдите выбросы с помощью метода z-отклонения и удалите их из данных, используйте логарифмический масштаб. Давайте сделаем послабление на **1 сигму** (возьмите 4 сигмы) в **правую сторону**.

Выведите таблицу с полученными выбросами и оцените, с каким возрастом соискатели попадают под категорию выбросов?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Чтение данных
file_path = '/home/sexxlexx/Desktop/SKILL/dst_cleaned_no_missing.csv'
data = pd.read_csv(file_path, sep=",")

# Удаление записи с возрастом 100
data = data[data['Возраст'] != 100]

# Построение распределения возраста в логарифмическом масштабе
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
log_age = np.log(data['Возраст'] + 1)  # Логарифмическое преобразование
histplot = sns.histplot(log_age, bins=30, ax=ax)
histplot.axvline(log_age.mean(), color='k', lw=2, label='Среднее')
histplot.axvline(log_age.mean() + 4 * log_age.std(), color='k', ls='--', lw=2, label='+4σ')
histplot.axvline(log_age.mean() - 3 * log_age.std(), color='k', ls='--', lw=2, label='-3σ')
histplot.set_title('Log Age Distribution')
plt.legend()
plt.tight_layout()
plt.savefig('/home/sexxlexx/Desktop/SKILL/log_age_distribution_v2.png')
plt.show()

# Функция для поиска выбросов
def outliers_z_score_mod(data, feature, left=3, right=4, log_scale=False):
    if log_scale:
        x = np.log(data[feature] + 1)
    else:
        x = data[feature]
    mu = x.mean()
    sigma = x.std()
    lower_bound = mu - left * sigma
    upper_bound = mu + right * sigma
    outliers = data[(x < lower_bound) | (x > upper_bound)]
    cleaned = data[(x >= lower_bound) & (x <= upper_bound)]
    return outliers, cleaned

# Поиск выбросов и очищение данных
outliers, cleaned_data = outliers_z_score_mod(data, 'Возраст', left=3, right=4, log_scale=True)

# Вывод результата
print(f"Число выбросов: {outliers.shape[0]}")
print("Пример выбросов:")
print(outliers[['Возраст']].head())

# Сохранение очищенных данных (опционально)
output_path = '/home/sexxlexx/Desktop/SKILL/dst_cleaned_no_age_outliers.csv'
cleaned_data.to_csv(output_path, index=False)

Распределение:

Распределение асимметрично и немного скошено вправо. Это видно по правому "хвосту", который более вытянут.
Большинство значений сосредоточено в диапазоне 3.0–3.7 (в логарифмическом масштабе), что соответствует возрасту примерно от 20 до 40 лет в исходных данных.
Среднее значение:

Черная линия, проходящая через центр (логарифм среднего возраста), соответствует основной массе данных.
Границы метода трёх сигм:

Левая граница (-3σ) расположена близко к минимальным значениям. Это означает, что практически нет выбросов с экстремально низким возрастом.
Правая граница (+4σ) расположена далеко вправо, позволяя учесть допустимые высокие значения возраста. Однако данные, выходящие за эту границу, считаются выбросами.
Выбросы:

Выбросы сосредоточены справа от линии +4σ. Это люди с экстремально высоким возрастом, которые аномально выделяются из общей выборки.
Выводы:
Распределение логарифмического возраста позволяет лучше визуализировать и обнаруживать выбросы.
Основная масса возрастов находится в пределах 20–40 лет (в исходной шкале).
Выбросы сосредоточены среди людей с крайне высоким возрастом (выше правой границы +4σ). Эти данные нужно будет очистить для корректного анализа.